# AI Avatar Studio — Google Colab T4 GPU Worker (標準版)

適用 **免費 T4 GPU**，提供：
1. **聲音克隆 (`POST /api/tts`)**：使用 **F5-TTS**（自動裁切 4.5 秒乾淨參考音檔 + Whisper 自動轉錄 `ref_text` + 斷句優化，避免開頭夾雜英文與不自然停頓）
2. **人物影片生成 (`POST /api/lipsync`)**：
   * **單張照片** $\rightarrow$ **SadTalker**（自然轉頭、眨眼、臉部表情 + 對嘴）
   * **既有影片** $\rightarrow$ **Wav2Lip-GAN**（保留原影片動作 + 重新對嘴）

In [ ]:
#@title 1. 檢查 GPU 與安裝基礎套件 (約 2~3 分鐘)
!nvidia-smi
!pip install -q fastapi uvicorn python-multipart f5-tts openai-whisper soundfile pydub yt-dlp
!apt-get -qq update && apt-get -qq install -y ffmpeg

# 下載 cloudflared (免註冊即可建立公開 HTTPS Tunnel)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
print("✅ 基礎套件安裝完成")

In [ ]:
#@title 2. 下載 SadTalker (照片動態頭像) 與 Wav2Lip (影片對嘴) 模型
import os

# --- A. Wav2Lip (用於影片分身對嘴) ---
if not os.path.exists("/content/Wav2Lip"):
    !git clone -q https://github.com/Rudrabha/Wav2Lip.git /content/Wav2Lip
    !pip install -q librosa==0.9.2 opencv-python-headless scipy tqdm numba

os.makedirs("/content/Wav2Lip/checkpoints", exist_ok=True)
os.makedirs("/content/Wav2Lip/face_detection/detection/sfd", exist_ok=True)
if not os.path.exists("/content/Wav2Lip/checkpoints/wav2lip_gan.pth"):
    !wget -q -O /content/Wav2Lip/checkpoints/wav2lip_gan.pth "https://huggingface.co/Nekochu/Wav2Lip/resolve/main/wav2lip_gan.pth"
if not os.path.exists("/content/Wav2Lip/face_detection/detection/sfd/s3fd.pth"):
    !wget -q -O /content/Wav2Lip/face_detection/detection/sfd/s3fd.pth "https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth"

# --- B. SadTalker (用於照片轉動態說話影片：轉頭、眨眼、表情) ---
if not os.path.exists("/content/SadTalker"):
    !git clone -q https://github.com/OpenTalker/SadTalker.git /content/SadTalker
    !cd /content/SadTalker && bash scripts/download_models.sh

# 核心相依套件獨立安裝（避免 basicsr 編譯失敗中斷整個安裝）
!pip install -q yacs kornia resampy safetensors facexlib imageio-ffmpeg
!sed -i 's/np.float)/float)/g' /content/SadTalker/src/face3d/util/my_awing_arch.py 2>/dev/null || true
!sed -i 's/np.int)/int)/g' /content/SadTalker/src/face3d/util/my_awing_arch.py 2>/dev/null || true

print("✅ SadTalker + Wav2Lip 模型準備完成")

In [ ]:
#@title 3. 啟動 FastAPI GPU Worker 並產生公開連線網址
import importlib.util
import os
import re
import shutil
import subprocess
import threading
import time
import uuid
from pathlib import Path
import torch
import whisper
import uvicorn
from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse

API_TOKEN = ""  #@param {type:"string"}

WORK_DIR = Path("/content/worker_tmp")
WORK_DIR.mkdir(parents=True, exist_ok=True)

whisper_model = None
def get_whisper():
    global whisper_model
    if whisper_model is None:
        whisper_model = whisper.load_model("base")
    return whisper_model

app = FastAPI(title="Colab AI Avatar GPU Worker")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

def verify_token(authorization: str | None):
    if API_TOKEN.strip():
        expected = f"Bearer {API_TOKEN.strip()}"
        if authorization != expected:
            raise HTTPException(status_code=401, detail="Invalid API token")

@app.get("/health")
def health(authorization: str | None = Header(default=None)):
    verify_token(authorization)
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    return {"status": "ok", "gpu": gpu_name, "engines": ["F5-TTS", "SadTalker", "Wav2Lip-GAN"]}

@app.post("/api/tts")
async def tts_clone(
    text: str = Form(...),
    language: str = Form("zh"),
    speed: float = Form(1.0),
    ref_text: str = Form(""),
    ref_audio: UploadFile = File(...),
    engine: str = Form("standard"),
    authorization: str | None = Header(default=None),
):
    verify_token(authorization)
    req_id = uuid.uuid4().hex[:8]
    raw_ref = WORK_DIR / f"{req_id}_raw_{ref_audio.filename}"
    wav_ref = WORK_DIR / f"{req_id}_ref.wav"
    out_dir = WORK_DIR / f"{req_id}_tts_out"
    out_dir.mkdir(exist_ok=True)

    raw_ref.write_bytes(await ref_audio.read())
    # 裁切為 4.5 秒乾淨人聲並去除靜音，避免 F5-TTS 邊界切詞錯位漏出英文
    subprocess.run([
        "ffmpeg", "-y", "-i", str(raw_ref),
        "-af", "silenceremove=start_periods=1:start_silence=0.05:start_threshold=-40dB,areverse,silenceremove=start_periods=1:start_silence=0.05:start_threshold=-40dB,areverse",
        "-t", "4.5",
        "-ar", "24000", "-ac", "1", str(wav_ref)
    ], check=True)

    resolved_ref_text = ref_text.strip()
    if not resolved_ref_text:
        try:
            res = get_whisper().transcribe(str(wav_ref), fp16=torch.cuda.is_available())
            resolved_ref_text = (res.get("text") or "").strip()
        except Exception:
            resolved_ref_text = ""
    if resolved_ref_text and resolved_ref_text[-1] not in "。！？.!?,，":
        resolved_ref_text += "."

    # 將換行與多餘空白整理為流暢句式，減少中間不自然斷句
    clean_text = re.sub(r"\s+", " ", text.replace("\n", "，")).strip()
    if clean_text and clean_text[-1] not in "。！？.!?,，":
        clean_text += "。"

    cmd = [
        "f5-tts_infer-cli",
        "--model", "F5TTS_v1_Base",
        "--ref_audio", str(wav_ref),
        "--gen_text", clean_text,
        "--speed", str(speed),
        "--nfe_step", "32",
        "--cross_fade_duration", "0.15",
        "--remove_silence",
        "--output_dir", str(out_dir),
        "--output_file", "output.wav",
    ]
    if resolved_ref_text:
        cmd.extend(["--ref_text", resolved_ref_text])

    proc = subprocess.run(cmd, capture_output=True, text=True)
    out_wav = out_dir / "output.wav"
    if proc.returncode != 0 or not out_wav.exists():
        raise HTTPException(status_code=500, detail=f"TTS failed: {proc.stderr[-500:]}")

    return FileResponse(str(out_wav), media_type="audio/wav", filename="output.wav")

@app.post("/api/lipsync")
async def lipsync(
    avatar: UploadFile = File(...),
    audio: UploadFile = File(...),
    engine: str = Form("standard"),
    authorization: str | None = Header(default=None),
):
    verify_token(authorization)
    req_id = uuid.uuid4().hex[:8]
    ext = Path(avatar.filename or "face.png").suffix.lower() or ".png"
    face_path = WORK_DIR / f"{req_id}_face{ext}"
    raw_audio = WORK_DIR / f"{req_id}_audio_raw.wav"
    wav_audio = WORK_DIR / f"{req_id}_audio_16k.wav"
    out_mp4 = WORK_DIR / f"{req_id}_result.mp4"

    face_path.write_bytes(await avatar.read())
    raw_audio.write_bytes(await audio.read())
    subprocess.run(["ffmpeg", "-y", "-i", str(raw_audio), "-ar", "16000", "-ac", "1", str(wav_audio)], check=True)

    is_image = ext in {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    if is_image and os.path.exists("/content/SadTalker/inference.py"):
        sad_out = WORK_DIR / f"{req_id}_sad"
        sad_out.mkdir(exist_ok=True)
        cmd = [
            "python", "/content/SadTalker/inference.py",
            "--driven_audio", str(wav_audio),
            "--source_image", str(face_path),
            "--result_dir", str(sad_out),
            "--preprocess", "full",
            "--expression_scale", "1.15",
        ]
        if importlib.util.find_spec("gfpgan") is not None:
            cmd.extend(["--enhancer", "gfpgan"])

        proc = subprocess.run(cmd, cwd="/content/SadTalker", capture_output=True, text=True)
        mp4s = list(sad_out.rglob("*.mp4"))
        if proc.returncode == 0 and mp4s:
            shutil.copyfile(mp4s[0], out_mp4)
            return FileResponse(str(out_mp4), media_type="video/mp4", filename="result.mp4")
        else:
            print("SadTalker fallback to Wav2Lip:", proc.stderr[-300:])

    cmd = [
        "python", "/content/Wav2Lip/inference.py",
        "--checkpoint_path", "/content/Wav2Lip/checkpoints/wav2lip_gan.pth",
        "--face", str(face_path),
        "--audio", str(wav_audio),
        "--outfile", str(out_mp4),
        "--nosmooth",
    ]
    proc = subprocess.run(cmd, cwd="/content/Wav2Lip", capture_output=True, text=True)
    if proc.returncode != 0 or not out_mp4.exists():
        raise HTTPException(status_code=500, detail=f"LipSync failed: {proc.stderr[-500:]}")

    return FileResponse(str(out_mp4), media_type="video/mp4", filename="result.mp4")

@app.post("/api/youtube-audio")
async def youtube_audio(
    url: str = Form(...),
    start_sec: float = Form(0.0),
    duration_sec: float = Form(12.0),
    authorization: str | None = Header(default=None),
):
    verify_token(authorization)
    req_id = uuid.uuid4().hex[:8]
    raw_tpl = str(WORK_DIR / f"{req_id}_yt.%(ext)s")
    out_wav = WORK_DIR / f"{req_id}_yt_trim.wav"

    dl_proc = subprocess.run(
        ["yt-dlp", "-f", "bestaudio/best", "--no-playlist", "-o", raw_tpl, url.strip()],
        capture_output=True,
        text=True,
    )
    downloaded = next(WORK_DIR.glob(f"{req_id}_yt.*"), None)
    if dl_proc.returncode != 0 or not downloaded:
        raise HTTPException(status_code=400, detail=f"yt-dlp failed: {dl_proc.stderr[-300:]}")

    ff_proc = subprocess.run(
        [
            "ffmpeg", "-y",
            "-ss", str(max(0.0, start_sec)),
            "-t", str(min(45.0, max(3.0, duration_sec))),
            "-i", str(downloaded),
            "-ar", "24000", "-ac", "1",
            str(out_wav),
        ],
        capture_output=True,
        text=True,
    )
    downloaded.unlink(missing_ok=True)
    if ff_proc.returncode != 0 or not out_wav.exists():
        raise HTTPException(status_code=500, detail=f"ffmpeg trim failed: {ff_proc.stderr[-300:]}")

    return FileResponse(str(out_wav), media_type="audio/wav", filename="youtube_voice.wav")

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8899, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

print("🚀 正在啟動 Cloudflare Tunnel，請稍候...")
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8899"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
for line in tunnel.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        print("\n" + "=" * 60)
        print("✅ Colab GPU Worker 已上線！請複製以下網址貼到本機設定：")
        print(match.group(0))
        print("=" * 60 + "\n")